# GBM drift and volatility with Nested Sampling

One-year estimation windows, weekly log-returns, nested sampling posterior for annualized `mu` and `sigma`.

In [31]:
import dynesty
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from dynesty import plotting as dyplot
from dynesty.utils import quantile

In [32]:
stock_name = 'AAPL'
start_year = 2000
end_year = 2023

return_window = 5
dt = return_window / 252

mu_min, mu_max = -2.0, 2.0
sigma_min, sigma_max = 0.01, 3.0

nlive = 300
dlogz = 0.05
random_seed = 123
example_year = 2023

In [ ]:
downloaded = yf.download(
    stock_name,
    start=f'{start_year}-01-01',
    end=f'{end_year + 1}-01-01',
    auto_adjust=False,
    progress=False,
)

prices = downloaded['Close']
if isinstance(prices, pd.DataFrame):
    prices = prices.iloc[:, 0]

prices = prices.dropna()
prices.head()

In [ ]:
def get_year_returns(prices, year, return_window):
    year_prices = prices.loc[prices.index.year == year].dropna()
    weekly_prices = year_prices.iloc[::return_window]
    log_returns = np.log(weekly_prices / weekly_prices.shift(1)).dropna().to_numpy()
    return year_prices, log_returns


def prior_transform(u):
    x = np.array(u)
    x[0] = mu_min + x[0] * (mu_max - mu_min)
    x[1] = sigma_min + x[1] * (sigma_max - sigma_min)
    return x


def make_loglike(log_returns):
    def loglike(theta):
        mu, sigma = theta
        mean = (mu - 0.5 * sigma**2) * dt
        std = sigma * np.sqrt(dt)
        residual = (log_returns - mean) / std
        return -0.5 * np.sum(residual**2 + np.log(2 * np.pi * std**2))

    return loglike

In [ ]:
def posterior_summary(results):
    samples = results.samples
    weights = results.importance_weights()
    row = {}

    for name, values in [('mu', samples[:, 0]), ('sigma', samples[:, 1])]:
        low, median, high = quantile(values, [0.025, 0.5, 0.975], weights=weights)
        mean = np.average(values, weights=weights)
        std = np.sqrt(np.average((values - mean)**2, weights=weights))

        row[f'{name}_mean'] = mean
        row[f'{name}_std'] = std
        row[f'{name}_q025'] = low
        row[f'{name}_median'] = median
        row[f'{name}_q975'] = high

    return row


def run_nested_for_year(year):
    year_prices, log_returns = get_year_returns(prices, year, return_window)
    loglike = make_loglike(log_returns)

    sampler = dynesty.NestedSampler(
        loglike,
        prior_transform,
        ndim=2,
        nlive=nlive,
        rstate=np.random.default_rng(random_seed + year),
    )
    sampler.run_nested(dlogz=dlogz, print_progress=False)

    results = sampler.results
    row = posterior_summary(results)
    row['year'] = year
    row['price_observations'] = len(year_prices)
    row['return_observations'] = len(log_returns)
    row['logz'] = results.logz[-1]
    row['logzerr'] = results.logzerr[-1]

    return row, results

In [ ]:
rows = []
yearly_results = {}

for year in range(start_year, end_year + 1):
    print(f'Running {year}')
    row, results = run_nested_for_year(year)
    rows.append(row)
    yearly_results[year] = results

yearly_summary = pd.DataFrame(rows).sort_values('year').reset_index(drop=True)
yearly_summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

mu_yerr = [
    yearly_summary['mu_median'] - yearly_summary['mu_q025'],
    yearly_summary['mu_q975'] - yearly_summary['mu_median'],
]
axes[0].errorbar(yearly_summary['year'], yearly_summary['mu_median'], yerr=mu_yerr, fmt='o-', capsize=3)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_ylabel('annualized drift')
axes[0].set_title(f'{stock_name}: yearly posterior for mu')

sigma_yerr = [
    yearly_summary['sigma_median'] - yearly_summary['sigma_q025'],
    yearly_summary['sigma_q975'] - yearly_summary['sigma_median'],
]
axes[1].errorbar(yearly_summary['year'], yearly_summary['sigma_median'], yerr=sigma_yerr, fmt='o-', capsize=3)
axes[1].set_xlabel('year')
axes[1].set_ylabel('annualized volatility')
axes[1].set_title(f'{stock_name}: yearly posterior for sigma')

fig.tight_layout()

In [ ]:
results = yearly_results[example_year]

fg, axs = dyplot.cornerplot(
    results,
    labels=[r'$\mu$', r'$\sigma$'],
    show_titles=True,
)